# Tính năng 1 – Trang **Dữ liệu huấn luyện**

Trang này cho thấy dữ liệu mà mô hình học: phễu làm sạch, cách chia train/valid/test và bảng dữ liệu tải về được. Notebook đi lại các bước phía sau, theo mạch **vấn đề → hậu quả nếu để nguyên → lựa chọn → lý do → cái giá**:

1. Dữ liệu thô có những vấn đề gì (đếm lại trên file gốc).
2. **Tin sinh tự động** – phát hiện quan trọng nhất – và vì sao phải loại.
3. Quy tắc chuẩn hóa lương chạy trên ví dụ thật.
4. Phễu làm sạch và chia tập **theo công ty**.

Chi tiết đầy đủ về từng nhãn (vai trò, cấp bậc, kỹ năng) và độ chính xác của bộ gán nhãn nằm ở notebook `gan_nhan_du_lieu.ipynb`. Mã nguồn: `src/career_market/salary_dataset.py`, `pages/dataset.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Ba nguồn thô và những vấn đề của chúng

In [ ]:
import re
import unicodedata
from career_market.paths import data_root

RAW = data_root() / "raw"
topcv = pd.read_csv(RAW / "topcv_jobs.csv")
posts = pd.read_csv(RAW / "vietnam_it_job_posts_v1.csv")
cb = pd.read_csv(RAW / "vietnamese-job-posting.csv")
display(pd.DataFrame({"Bộ dữ liệu": ["TopCV 2026", "IT Job Posts 2026", "CareerBuilder 2023"],
                      "Số dòng": [len(topcv), len(posts), len(cb)], "Số cột": [topcv.shape[1], posts.shape[1], cb.shape[1]]}))

raw_salary = pd.concat([topcv["salary"], posts["salary"], cb["salary"]]).fillna("").astype(str)
# Dữ liệu thô viết dấu tiếng Việt theo hai kiểu Unicode (dựng sẵn và tổ hợp): chuẩn hóa NFC trước khi so khớp.
salary = raw_salary.map(lambda v: unicodedata.normalize("NFC", v)).str.lower()
problems = {
    "Ghi 'thỏa thuận / thương lượng / cạnh tranh'": salary.str.contains("thỏa thuận|thoả thuận|thương lượng|cạnh tranh"),
    "Dấu tiếng Việt kiểu tổ hợp (nhìn giống, khác byte)": raw_salary != raw_salary.map(lambda v: unicodedata.normalize("NFC", v)),
    "Lỗi chính tả 'triẹu'": salary.str.contains("triẹu"),
    "Lương bằng USD": salary.str.contains(r"\$|usd"),
    "Lương theo năm": salary.str.contains(r"/\s*năm"),
    "Khoảng giả '1 triệu - 40 triệu'": salary.str.contains(r"^1 tri[ẹệ]u - 40 tri[ẹệ]u"),
}
pd.DataFrame({"Vấn đề": problems.keys(), "Số dòng": [int(m.sum()) for m in problems.values()]}).assign(
    **{"Tỉ lệ": lambda t: (t["Số dòng"] / len(salary) * 100).round(1).astype(str) + "%"})

**Khoảng lương giả** "1 – 40 triệu" là giá trị mặc định của biểu mẫu đăng tin. Trung điểm 20,5 triệu trông rất bình thường nên máy không báo lỗi mà lặng lẽ học sai – loại lỗi nguy hiểm nhất. Quy tắc: bỏ mọi khoảng có cận trên gấp hơn 3 lần cận dưới.

## 2. Phát hiện lớn nhất: 2.400 tin sinh tự động

**Dấu hiệu đầu tiên:** ngày đăng ở ba nền tảng TopCV, VietnamWorks, CareerViet rải đều 2018–2025 – thị trường thật không bao giờ đều như vậy. Ban đầu nhóm chỉ coi đây là cột hỏng. Điều tra tiếp cho thấy cả nhóm tin đó có chung các dấu hiệu dưới đây.

In [ ]:
from career_market.data.rules import is_synthetic

synthetic = posts["source_url"].map(is_synthetic)
print("Số tin có đường dẫn dạng …-hist…:", int(synthetic.sum()))
display(posts[synthetic]["source_platform"].value_counts().rename("Số tin theo nền tảng").to_frame())
display(pd.DataFrame({
    "Tin sinh tự động": [posts[synthetic]["requirements"].notna().mean() * 100],
    "Tin thật": [posts[~synthetic]["requirements"].notna().mean() * 100],
}, index=["% có mô tả yêu cầu"]))
posts[synthetic][["title", "source_url", "skills"]].head(5)

In [ ]:
years = pd.DataFrame({
    "Tin sinh tự động": posts[synthetic]["posted_at"].astype(str).str[:4].value_counts(),
    "Tin thật": posts[~synthetic]["posted_at"].astype(str).str[:4].value_counts(),
}).fillna(0).sort_index()
ax = years.plot.bar(figsize=(9, 3.5), color=["#f97316", "#2563eb"], title="Số tin theo năm đăng ghi trong dữ liệu")
ax.set(xlabel="", ylabel="Số tin");

- **Vấn đề:** 2.400 tin (đúng 800 mỗi nền tảng) có đường dẫn `…-hist…`, ngày đăng rải đều, kỹ năng ngẫu nhiên (Android Developer có Swift và React), không có mô tả. Chúng chiếm 54% số tin có nhãn trước khi loại.
- **Hậu quả nếu giữ:** tin giả dễ đoán hơn tin thật nên làm điểm đẹp giả – mô hình cũ có trung vị sai số 16,5% trên tập test lẫn tin sinh nhưng 18,0% trên riêng tin thật. Giống ôn thi bằng bộ đề do máy bịa.
- **Lựa chọn:** loại hẳn (thay vì chỉ sửa cột ngày hoặc đánh dấu).
- **Cái giá:** mất hơn nửa dữ liệu; tập test còn 288 tin.

## 3. Chuẩn hóa lương – mỗi quy tắc có lý do

| Dạng ghi | Quy tắc | Vì sao |
|---|---|---|
| "15 - 25 triệu" | trung điểm 20 | Nhà tuyển dụng hay nói "lương khoảng 20 triệu"; cận trên là mức trần |
| "Tới 30 triệu" / "Từ 20 triệu" | suy đầu còn lại bằng **hệ số cao/thấp của từng nguồn** (học trên train) | Mỗi trang có thói quen ghi khoảng khác nhau; hệ số cố định 0,7 ban đầu không đúng cho mọi trang |
| USD / theo năm | × tỷ giá năm dữ liệu / ÷ 12 | Tránh "1.000 USD" thành 1.000 triệu |
| Khoảng > 3 lần | loại | Một vị trí hiếm khi chênh quá 3 lần; khoảng rộng hơn là né công bố |

In [ ]:
import json
from career_market.data.rules import parse_salary
from career_market.paths import dataset_path

summary = json.loads(dataset_path().with_suffix(".json").read_text(encoding="utf-8"))
ratios = summary["one_sided_ratio"]
display(pd.Series(ratios, name="Hệ số cao/thấp (học trên train)").to_frame())

examples = ["15 - 25 triệu", "Lương: 10 Tr - 15 Tr VND", "$ 1,000-1,500 /tháng", "240tr-360tr ₫/năm", "1 triẹu - 40 triẹu/tháng", "Thỏa thuận"]
rows = [{"Chuỗi gốc": t, "Kết quả (thấp, cao, trung điểm)": parse_salary(t, 2026) or "loại khỏi nhãn"} for t in examples]
for text in ["Tới 30 triệu", "Từ 20 triệu"]:
    for source in ["IT Jobs 2026 (itviec)", "TopCV 2026"]:
        rows.append({"Chuỗi gốc": f"{text}  [{source}]", "Kết quả (thấp, cao, trung điểm)": parse_salary(text, 2026, ratio=ratios[source])})
pd.DataFrame(rows)

## 4. Phễu làm sạch và chia tập theo công ty

**Vì sao chia theo công ty?** Một công ty đăng hàng chục tin gần giống nhau. Chia ngẫu nhiên thì mô hình chỉ cần "nhớ" công ty là đoán đúng – như ôn đúng đề sẽ thi. Câu hỏi thật của người dùng là về **công ty mới**, nên tập test phải gồm công ty mô hình chưa từng gặp. Hàm băm SHA-256 trên tên công ty (seed 42) giúp kết quả chia luôn tái lập được.

In [ ]:
from career_market.data.dataset import load_dataset

funnel = summary["funnel"]
steps = pd.DataFrame({
    "Bước": ["Đọc ba nguồn (CareerBuilder chỉ nhóm CNTT)", "Sau khi loại tin sinh tự động", "Có lương đọc được", "Thuộc 23 vai trò CNTT", "Sau khử trùng lặp"],
    "Số tin": [funnel["raw"], funnel["raw"] - funnel["synthetic_removed"], funnel["has_salary"], funnel["it_role"], funnel["deduplicated"]],
})
steps["Tỉ lệ"] = (steps["Số tin"] / funnel["raw"] * 100).round(1).astype(str) + "%"
display(steps)

frame = load_dataset()
table = frame.groupby("split").agg(**{"Số tin": ("row_id", "size"), "Số công ty": ("company_key", "nunique")}).loc[["train", "valid", "test"]]
table["Tỉ lệ tin"] = (table["Số tin"] / len(frame) * 100).round(1).astype(str) + "%"
display(table)
groups = {s: set(frame.loc[frame["split"] == s, "company_key"]) for s in ["train", "valid", "test"]}
print("Công ty xuất hiện ở hai tập:", len(groups["train"] & groups["valid"]) + len(groups["train"] & groups["test"]) + len(groups["valid"] & groups["test"]))
print("Còn tin sinh tự động trong bộ dữ liệu sạch:", int(frame["source_url"].map(is_synthetic).sum()))

Hai biểu đồ dưới đây chính là hai biểu đồ trên trang Dữ liệu huấn luyện.

In [ ]:
charts.split_by_role(frame);

In [ ]:
charts.salary_distribution(frame);

## Kết luận
- Mỗi lỗi của dữ liệu thô có một quy tắc xử lý minh bạch, kiểm thử được.
- Bài học lớn nhất: *nhiều dữ liệu chỉ có giá trị khi dữ liệu là thật*. Loại 2.400 tin sinh tự động làm sai số tăng nhưng trung thực.
- Chia theo công ty là quyết định phương pháp quan trọng nhất: mọi con số đánh giá đều đo trên công ty chưa từng gặp.